In [ ]:
import os
import urllib.request
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Import the official SAM 1 tools
from segment_anything import sam_model_registry, SamPredictor

# --- 1. LOCAL SETUP ---
# This matches your local Windows structure
CHECKPOINT_PATH = "sam_vit_b_01ec64.pth"
checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

# Download weights if they aren't in your folder yet
if not os.path.exists(CHECKPOINT_PATH):
    print("Downloading SAM 1 weights (375MB)... please wait.")
    urllib.request.urlretrieve(checkpoint_url, CHECKPOINT_PATH)
    print("Download complete!")

image_dir = r"C:\Users\Joon\Desktop\ME_592_robotics_HW\6_test_images"  

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading SAM 1 onto {device}...")

# Load the Model
sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT_PATH)
sam.to(device=device)
predictor = SamPredictor(sam)

# --- 2. YOUR COORDINATES ---
data = {
    "pcd0800r.png": [271, 261, 323, 343],
    "pcd0801r.png": [227, 191, 350, 306],
    "pcd0802r.png": [219, 174, 351, 290],
    "pcd0803r.png": [211, 192, 363, 309],
    "pcd0804r.png": [205, 153, 381, 341],
    "pcd0805r.png": [137, 151, 396, 362]
}

# --- 3. THE EVALUATION LOOP ---
for filename, box in data.items():
    img_path = os.path.join(image_dir, filename)
    if not os.path.exists(img_path):
        print(f"⚠️ Skipping {filename} - Not found at {img_path}")
        continue

    print(f"Processing {filename}...")
    image = np.array(Image.open(img_path).convert("RGB"))
    predictor.set_image(image)
    
    # Points
    cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
    height = box[3] - box[1]
    point_coords = np.array([[cx, cy], [cx, cy - height*0.25], [cx, cy + height*0.25]])
    point_labels = np.array([1, 1, 1]) 

    masks_pts, scores_pts, _ = predictor.predict(
        point_coords=point_coords,
        point_labels=point_labels,
        multimask_output=False, 
    )

    # Box
    box_coords = np.array(box)
    masks_box, scores_box, _ = predictor.predict(
        box=box_coords[None, :], 
        multimask_output=False,
    )

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    fig.suptitle(f"File: {filename}", fontsize=16)

    axes[0].imshow(image)
    axes[0].imshow(masks_pts[0], alpha=0.5, cmap='autumn')
    axes[0].set_title(f"Points (Conf: {min(scores_pts[0], 1.0):.2f})")
    axes[0].axis('off')

    axes[1].imshow(image)
    axes[1].imshow(masks_box[0], alpha=0.5, cmap='winter')
    rect = plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], edgecolor='blue', facecolor='none', lw=2)
    axes[1].add_patch(rect)
    axes[1].set_title(f"Box (Conf: {min(scores_box[0], 1.0):.2f})")
    axes[1].axis('off')

    plt.show()

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Joon\Desktop\ME_592_robotics_HW\.venv_311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.